# Synthetic Data Generation with vLLM

This notebook generates synthetic reasoning data using vLLM for the MI300X Reasoning Booster project.

**Prerequisites:**
- vLLM server running on port 8001
- ROCm PyTorch installed
- OpenAI library for API client

In [ ]:
import json
from openai import OpenAI
from tqdm import tqdm
import random

# Connect to vLLM server
client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="dummy"  # vLLM doesn't require real API key
)

print("Connected to vLLM server")

In [ ]:
# Chain-of-Thought prompt for reasoning generation
cot_prompt = """Create a complex logical reasoning problem with a detailed Chain-of-Thought solution.
The problem should cover one of these topics: math, logic puzzles, code reasoning, or multi-step inference.
Each problem must include:
- Clear problem statement
- Step-by-step reasoning
- Final answer

Return ONLY valid JSON with this format:
{
  "question": "...",
  "reasoning": "...",
  "answer": "..."
}

Problem:"""

def generate_sample():
    """Generate one synthetic reasoning sample using vLLM"""
    try:
        response = client.chat.completions.create(
            model="unsloth/Llama-3.3-70B-Instruct",
            messages=[
                {"role": "user", "content": cot_prompt}
            ],
            temperature=0.8,
            top_p=0.95,
            max_tokens=512
        )
        
        generated = response.choices[0].message.content
        
        # Extract JSON from generated text
        start = generated.find("{")
        end = generated.rfind("}") + 1
        if start != -1 and end > start:
            json_str = generated[start:end]
            data = json.loads(json_str)
            return data
    except Exception as e:
        print(f"Error generating sample: {e}")
    
    return None

In [ ]:
# Configuration
num_samples = 100  # Increase to 1000-5000 for more data
samples = []

print(f"Generating {num_samples} synthetic reasoning samples...")

In [ ]:
# Generate samples with progress tracking
for i in tqdm(range(num_samples), desc="Generating samples"):
    sample = generate_sample()
    if sample:
        samples.append(sample)
    
    # Save progress every 10 samples
    if (i + 1) % 10 == 0:
        with open("data/synthetic_reasoning.jsonl", "w") as f:
            for s in samples:
                f.write(json.dumps(s) + "\n")
        print(f"\nSaved {len(samples)} samples to data/synthetic_reasoning.jsonl")

In [ ]:
# Final save
with open("data/synthetic_reasoning.jsonl", "w") as f:
    for s in samples:
        f.write(json.dumps(s) + "\n")

print(f"\nGeneration complete! Generated {len(samples)} valid samples.")
print(f"Saved to data/synthetic_reasoning.jsonl")

In [ ]:
# Preview a sample
if samples:
    print("\nSample:")
    print(json.dumps(samples[0], indent=2))

In [ ]:
# Convert to training format for LoRA SFT
training_data = []
for s in samples:
    training_data.append({
        'conversations': [
            {'from': 'human', 'value': s['question']},
            {'from': 'gpt', 'value': f"Reasoning: {s['reasoning']}\n\nAnswer: {s['answer']}"}
        ]
    })

with open("data/final/train.jsonl", "w") as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')

print(f"Converted {len(training_data)} samples to training format")
print("Saved to data/final/train.jsonl")